<a href="https://colab.research.google.com/github/naheejeong/cs182-final-project/blob/main/00_llama_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# # hardware check - GPU & memory assigned
!nvidia-smi

Sat Nov 21 06:03:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   76C    P0             36W /   72W |   20760MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Connect Google Drive to the Colab machine so results are saved to Drive.
from google.colab import drive
drive.mount('/content/drive')

# Setup commands (! = system command, % = notebook command)
%cd /content

# Download the InjecAgent benchmark (attack cases and evaluation code) from GitHub.
# --> this makes a new folder named "InjecAgent" in /content.
!git clone https://github.com/uiuc-kang-lab/InjecAgent.git
%cd /content/InjecAgent

# Install the 6 required packages (4 from requirements.txt + together and nltk).
!pip install -q -r requirements.txt together nltk

# Create a results folder in Google Drive (My Drive/cs182/results).
!mkdir -p /content/drive/MyDrive/cs182/results

# Create shortcut: InjecAgent/results -> My Drive/cs182/results, so outputs are saved to Drive.
!ln -sfn /content/drive/MyDrive/cs182/results results

# Provide environment variables needed by the InjecAgent code.
import os
os.environ["PYTHONPATH"] = "/content/InjecAgent"
os.environ["OPENAI_API_KEY"] = "unused" # placeholder; we never use!

!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
fatal: destination path 'InjecAgent' already exists and is not an empty directory.
/content/InjecAgent
asset  data  LICENCE  README.md  requirements.txt  results  src


In [ ]:
# Log in to HF using the token stored in Secrets. Token name: HF_TOKEN, username: naheejeong
from google.colab import userdata
from huggingface_hub import login, whoami, hf_hub_download
login(token=userdata.get('HF_TOKEN'))
print("Logged in as:", whoami()['name'])

# Testing Llama access - download one tiny file from the model's page.
path = hf_hub_download("meta-llama/Llama-3.1-8B-Instruct", "config.json")
print(path)
print(open(path).read())    # printing the config file

Logged in as: naheejeong
/root/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659/config.json
{
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 8.0,
    "low_freq_factor": 1.0,
    "high_freq_factor": 4.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.42.3",
  "use_cache": true,
  "vocab_si

In [ ]:
# A little background in InjecAgent:
# InjecAgent doesn’t care which model it tests, but it requires that each model is wrapped in a class with three functions.
# Below code shows the template its authors wrote:
!head -n 12 /content/InjecAgent/src/models.py

import os
from src.utils import get_response_text
import time
class BaseModel:
    def __init__(self):
        self.model = None

    def prepare_input(self, sys_prompt,  user_prompt_filled):
        raise NotImplementedError("This method should be overridden by subclasses.")

    def call_model(self, model_input):
        raise NotImplementedError("This method should be overridden by subclasses.")


In [ ]:
from transformers import AutoTokenizer

model_name = "meta-llama/Llama-3.1-8B-Instruct"
tok = AutoTokenizer.from_pretrained(model_name)   # downloads Llama's tokenizer

# tok("some text") returns:
# {
#  "input_ids":      [128000, 2028, 374, ...],
#  "attention_mask": [1, 1, 1, ...]
# }
ids = tok("This is a test input Meow :3")["input_ids"] # --> E.g., [128000, 2028, 374, ...]. Turning text into ids.

print(ids)
print(tok.convert_ids_to_tokens(ids))
print(tok.decode(ids))

[128000, 2028, 374, 264, 1296, 1988, 2206, 363, 551, 18]
['<|begin_of_text|>', 'This', 'Ġis', 'Ġa', 'Ġtest', 'Ġinput', 'ĠMe', 'ow', 'Ġ:', '3']
<|begin_of_text|>This is a test input Meow :3


In [ ]:
from torch import bfloat16
from transformers import AutoModelForCausalLM
from transformers import __version__ as transformers_version

print("transformers version:", transformers_version)

# Download Llama's weights (~16 GB) and load them onto the GPU at half precision
# FYI - Half precision just means each parameter is stored in 16 bits instead of 32,
# so the memory is halved (~16 GB instead of ~32 GB).
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=bfloat16,   # Meta published this model's weights in bfloat16 (half precision)
    device_map="auto",    # put the model on GPU automatically
)

!nvidia-smi

In [ ]:
print(model.device) # Should print "cuda:0"
# print(tok.chat_template)

In [ ]:
# Chat format check
# Recreating the example here for sanity check: https://dev.meta.ai/llama/docs/model-cards-and-prompt-formats/llama3_1#:~:text=%3E%7B%7B%20user_message%7D%7D-,Instruct%20Model%20Prompt,-The%20format%20for

# ---------------------------------------------------------------------------
# Chat format: how we describe a prompt to a chat model.
#
# A prompt is written as a list of messages. Each message is a dictionary with:
#   "role"    - who the text comes from
#   "content" - the text itself
#
# Roles used here:
#   "system" - standing instructions that set the model's behaviour
#   "user"   - the actual request
# (A third role, "assistant", holds the model's own replies in longer chats.)
#
# This list format is a shared convention across chat models. The model never
# sees the list itself: the tokenizer's chat template converts it into one
# string with the model's own special marker tokens (see the next lines).
#
# In our project, InjecAgent supplies both texts: its agent instructions go in
# the "system" message, and the test case (user task + tool output containing
# the attack) goes in the "user" message.
# ---------------------------------------------------------------------------
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"},
]

# Converts the messages list into the single string, with marker tokens, that Llama 3.1 was trained to read
# add_generation_prompt=True cues Llama “your turn to answer”.
# tokenize=False returns the result as readable text.
prompt_text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
print(prompt_text)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the capital of France?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [ ]:
from torch import no_grad

# 1. Format the messages AND turn them into token IDs
inputs = tok.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,   # returns a dictionary with two entries: "input_ids" and "attention_mask".
    return_tensors="pt",   # return token ids as PyTorch tensors
).to(model.device)    # move them to the GPU
print("prompt length in tokens:", inputs["input_ids"].shape[-1])

# 2. Let Llama generate the continuation
with no_grad():   # inference, so no gradient calculation needed
    out = model.generate(
        **inputs,   # unpack "input_ids" and "attention_mask"
        max_new_tokens=512,
        do_sample=False, # no randomness: always pick the most likely token (same as the InjecAgent paper)
        pad_token_id=tok.eos_token_id, # use eos token as the padding token
    )
print("prompt + reply length in tokens:", out.shape[-1])

# 3. Keep only the new tokens (the reply) and turn them back into text.
all_tokens = out[0]
prompt_length = inputs["input_ids"].shape[-1]
reply_tokens = all_tokens[prompt_length:]
reply = tok.decode(reply_tokens, skip_special_tokens=True)
print(reply)

prompt length in tokens: 48
prompt + reply length in tokens: 56
The capital of France is Paris.
